In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from adjustText import adjust_text

from vpei.epistemic_consistency.results_utils import (
    compute_stats_from_experimental_results,
    load_models_experiments_results,
)
from vpei.common_utils import trim_model_names
from vpei.models import MODELS, MODELS_WITH_REASON_OFF

models = MODELS_WITH_REASON_OFF

experiments_types_and_names_to_load = {
    "comparative_experiment_with_ground_truth": [
        "code",
        "logical_reasoning",
        "math_proofs",
        "physics_problems",
        "factual_vs_false_statement_detection",
    ],
}

experimental_results_path = '~/repos/epistemic_consistency_paper/experimental_results'

In [ ]:
# Compute mean political bias (log_odds) per model across the 5 ground-truth categories
bias_stats_df = compute_stats_from_experimental_results(
    models,
    experiments_types_and_names_to_load,
    experimental_results_path=experimental_results_path,
)

bias_per_model = (
    bias_stats_df
    .groupby('model_name')['log_odds']
    .mean()
    .reset_index()
    .rename(columns={'log_odds': 'mean_bias'})
)
print(f"Models with bias data: {len(bias_per_model)}")
bias_per_model.head()

In [ ]:
# Compute mean accuracy (model_guessed_right) per model across the 5 ground-truth categories
all_results_df = load_models_experiments_results(
    models,
    experiments_types_and_names_to_load,
    experimental_results_path=experimental_results_path,
)

accuracy_per_model = (
    all_results_df
    .groupby('model_name')['model_guessed_right']
    .mean()
    .reset_index()
    .rename(columns={'model_guessed_right': 'mean_accuracy'})
)
print(f"Models with accuracy data: {len(accuracy_per_model)}")
accuracy_per_model.head()

In [ ]:
# Merge bias and accuracy, compute short display names
merged = bias_per_model.merge(accuracy_per_model, on='model_name', how='inner')
merged['short_name'] = trim_model_names(merged['model_name'].tolist())

print(f"Models in merged dataset: {len(merged)}")
merged[['short_name', 'mean_bias', 'mean_accuracy']].sort_values('mean_bias')

In [ ]:
# Scatter plot: accuracy vs political bias
x = merged['mean_bias'].to_numpy(dtype=float)
y = merged['mean_accuracy'].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(x, y, s=80, color='C2', zorder=3)

texts = []
for _, row in merged.iterrows():
    texts.append(ax.text(row['mean_bias'], row['mean_accuracy'], row['short_name'], fontsize=9))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

slope, intercept = np.polyfit(x, y, 1)
r_value, p_value = stats.pearsonr(x, y)
x_range = np.linspace(x.min(), x.max(), 200)
ax.plot(
    x_range,
    slope * x_range + intercept,
    color='firebrick',
    linewidth=1.5,
    linestyle='--',
    label=f'r = {r_value:.2f},  p = {p_value:.3f}  (n={len(merged)})',
)

ax.axvline(0, color='gray', linewidth=2, linestyle='--')
ax.set_xlabel('Political Bias — Mean Log Odds\n(comparative_experiment_with_ground_truth, 5 categories)', fontsize=13)
ax.set_ylabel('Mean Accuracy (model_guessed_right)', fontsize=13)
ax.set_title(
    'Model Accuracy vs. Political Bias\n'
    'Person-attribution experiments — pairwise ground-truth selection',
    fontsize=14,
    fontweight='bold',
)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
plt.tight_layout()
# fig.savefig('./figures/scatterplot_accuracy_vs_person_attribution_bias.png', dpi=300, bbox_inches='tight')
plt.show()